<a href="https://colab.research.google.com/github/haydin94githu/aydin-bist-robotu/blob/main/bist_robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
# from google.colab import files

# =========================
# BIST HİSSE LİSTESİ
# =========================
!pip install tradingview-screener -q

from tradingview_screener import Query, col

def get_bist_symbols_from_tradingview():
    try:
        _, df_tv = (
            Query()
            .select("name", "exchange", "type")
            .where(
                col("exchange") == "BIST",
                col("type") == "stock"
            )
            .limit(1000)
            .get_scanner_data()
        )

        symbols = df_tv["name"].dropna().unique().tolist()
        symbols = [s.replace("BIST:", "").strip() for s in symbols]
        symbols = sorted(list(set(symbols)))



    except Exception as e:
        print("TradingView liste çekme hatası:", e)
        return []

symbols = """
A1CAP ACSEL ADEL ADESE AEFES AFYON AGESA AGHOL AGROT AHGAZ AKBNK AKCNS AKENR AKFGY AKFIS AKGRT AKMGY AKSA AKSEN AKSGY AKSUE AKYHO ALARK ALBRK ALCAR ALCTL ALFAS ALGYO ALKA ALKIM ALMAD ALTNY ANELE ANGEN ANHYT ANSGR ARASE ARCLK ARDYZ ARENA ARSAN ARTMS ASELS ASTOR ASUZU ATAGY ATAKP ATATP AVGYO AVHOL AVOD AYCES AYDEM AYEN AYES AYGAZ AZTEK BAGFS BAHKM BAKAB BALAT BANVT BARMA BASCM BEGYO BERA BEYAZ BFREN BIENY BIGCH BIMAS BINBN BINHO BIOEN BIZIM BJKAS BLCYT BOBET BORLS BORSK BOSSA BRISA BRKSN BRLSM BRMEN BRYAT BSOKE BTCIM BUCIM BURCE BURVA BVSAN BYDNR CANTE CASA CCOLA CELHA CEMAS CEMTS CEOEM CIMSA CLEBI CMBTN CONSE COSMO CRDFA CRFSA CUSAN CVKMD CWENE DAGHL DAGI DAPGM DARDL DCTTR DENGE DERHL DESA DESPC DEVA DGATE DGGYO DGNMO DIRIT DITAS DMRGD DOAS DOBUR DOCO DOGUB DOHOL DOKTA DURDO DYOBY EBEBK ECILC ECZYT EDATA EDIP EFORC EGEEN EGEPO EGGUB EGPRO EGSER EKGYO EKIZ ELITE EMKEL ENERY ENJSA ENKAI ENSRI ENTRA EPLAS ERBOS EREGL ERSU ESCAR ESCOM EUPWR EUREN EUYO FADE FENER FLAP FMIZP FONET FORMT FORTE FROTO GARAN GARFA GEDIK GEDZA GENIL GESAN GLBMD GLCVY GLRYH GLYHO GMTAS GOKNR GOLTS GOODY GOZDE GRSEL GRTHO GSDDE GSDHO GSRAY GUBRF GWIND HALKB HATEK HATSN HEDEF HEKTS HKTM HLGYO HOROZ HRKET HTTBT HUBVC HUNER ICBCT IDEAS IDGYO IEYHO IHAAS IHEVA IHGZT IHLAS IHLGM IHYAY INDES INFO INGRM INTEM INVEO INVES IPEKE ISATR ISBIR ISCTR ISDMR ISFIN ISGSY ISKPL ISMEN ISSEN IZENR IZFAS IZINV IZMDC JANTS KAPLM KAREL KARSN KARTN KATMR KAYSE KCAER KCHOL KERVN KFEIN KGYO KLGYO KLKIM KLSER KLRHO KMPUR KONKA KONTR KONYA KOPOL KORDS KOZAA KOZAL KRDMA KRDMB KRDMD KRGYO KRONT KRPLS KRSTL KRVGD KSTUR KTLEV KUYAS KZBGY LIDER LINK LKMNH LOGO LRSHO LUKSK LYDHO MAALT MAGEN MAKIM MAKTK MANAS MARTI MAVI MEDTR MEGAP MEPET METRO MHRGY MIATK MNDRS MOBTL MOGAN MPARK MRGYO MRSHL MSGYO MTRKS MZHLD NATEN NETAS NIBAS NTGAZ NUHCM OBAMS ODAS ODINE OFSYM ONCSM ONRYT ORCAY ORGE OSTIM OTKAR OYAKC OYLUM OZGYO OZSUB PAGYO PAMEL PAPIL PARSN PASEU PATEK PCILT PEKGY PENGD PETKM PGSUS PINSU PKART PLTUR PNLSN POLHO POLTK PRDGS PRKAB PRKME PSDTC QUAGR QNBFB RALYH RAYSG REEDR RGYAS RNPOL RODRG RTALB RUBNS RYGYO RYSAS SAHOL SAMAT SANEL SANFM SARKY SASA SAYAS SDTTR SEGMN SEKFK SELEC SELGD SEYKM SILVR SISE SKBNK SMART SMRTG SNGYO SNICA SODSN SOKE SONME SRVGY SUMAS SUWEN TABGD TARKM TATEN TATGD TAVHL TBORG TCELL TEKTU TERA TGSAS THYAO TKFEN TKNSA TLMAN TMPOL TOASO TRCAS TRGYO TRILC TSGYO TSKB TTKOM TTRAK TUKAS TUPRS TUREX TURGG ULAS ULKER ULUSE UNLU USAK VAKBN VAKFN VAKKO VERTU VESBE VESTL VKGYO YAPRK YATAS YAYLA YBTAS YEOTK YGYO YKBNK YKSLN YONGA YUNSA YYLGD ZEDUR ZOREN ZRGYO
""".split()

symbols = sorted(list(set(symbols)))

print(f"Taranacak hisse sayısı: {len(symbols)}")
if len(symbols) == 0:
    print("Liste çekilemedi, eski liste kullanılacak.")

symbols = sorted(list(set(symbols)))

# =========================
# VERİ ÇEKME
# =========================
def download_data(symbol):
    ticker = symbol + ".IS"

    df = yf.download(
        ticker,
        period="2y",
        interval="1d",
        progress=False,
        auto_adjust=True,
        group_by="column"
    )

    if df is None or df.empty:
        return None

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    required = ["Open", "High", "Low", "Close", "Volume"]

    if not all(col in df.columns for col in required):
        return None

    df = df[required].copy()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna()

    if len(df) < 60:
        return None

    return df


def resample_ohlcv(df, rule):
    out = pd.concat({
        "Open": df["Open"].resample(rule).first(),
        "High": df["High"].resample(rule).max(),
        "Low": df["Low"].resample(rule).min(),
        "Close": df["Close"].resample(rule).last(),
        "Volume": df["Volume"].resample(rule).sum()
    }, axis=1)

    out = out.dropna()
    return out

# =========================
# İNDİKATÖRLER
# =========================
def rsi(series, length=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(length).mean()
    avg_loss = loss.rolling(length).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def mfi(df, length=14):
    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    mf = tp * df["Volume"]
    direction = tp.diff()

    pos = mf.where(direction > 0, 0).rolling(length).sum()
    neg = mf.where(direction < 0, 0).rolling(length).sum()

    return 100 - (100 / (1 + pos / neg.replace(0, np.nan)))

# =========================
# SİNYAL MOTORU
# =========================
def signal_engine(df):
    if df is None or len(df) < 30:
        return "YETERSİZ", 0,0

    close = df["Close"]
    high = df["High"]
    low = df["Low"]
    open_ = df["Open"]
    volume = df["Volume"]

    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()

    r = rsi(close, 14)
    mf = mfi(df, 14)

    macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
    macd_sig = macd.ewm(span=9, adjust=False).mean()

    vol_avg = volume.rolling(20).mean()

    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_low = bb_mid - 2 * bb_std
    bb_up = bb_mid + 2 * bb_std

    range_high = high.rolling(100).max()
    range_low = low.rolling(100).min()
    orta = (range_high + range_low) / 2

    onceki_dip = low.rolling(20).min().shift(1)

    if pd.isna(r.iloc[-1]) or pd.isna(mf.iloc[-1]) or pd.isna(vol_avg.iloc[-1]):
        return "YETERSİZ", 0,0

    c = float(close.iloc[-1])
    o = float(open_.iloc[-1])
    v = float(volume.iloc[-1])

    dip_supurme = low.iloc[-1] < onceki_dip.iloc[-1] and c > onceki_dip.iloc[-1]
    dip_bolge = c < orta.iloc[-1]
    bb_tepki = low.iloc[-1] <= bb_low.iloc[-1] and c > bb_low.iloc[-1]
    rsi_donus = r.iloc[-1] < 42 and r.iloc[-1] > r.iloc[-2]
    mfi_donus = mf.iloc[-1] < 45 and mf.iloc[-1] > mf.iloc[-2]
    hacim_tepki = v > vol_avg.iloc[-1] and c > o

    dip_score = 0
    dip_score += 2 if dip_supurme else 0
    dip_score += 1 if dip_bolge else 0
    dip_score += 1 if bb_tepki else 0
    dip_score += 1 if rsi_donus else 0
    dip_score += 1 if mfi_donus else 0
    dip_score += 1 if hacim_tepki else 0

    dipten_al = dip_score >= 3

    kurum_topluyor = c > o and v > vol_avg.iloc[-1] * 1.5 and mf.iloc[-1] > mf.iloc[-2]
    kurum_satiyor = c < o and v > vol_avg.iloc[-1] * 1.5 and mf.iloc[-1] < mf.iloc[-2]

    guvenli_al = (
        c > ema20.iloc[-1] and
        ema20.iloc[-1] > ema50.iloc[-1] and
        45 < r.iloc[-1] < 68 and
        macd.iloc[-1] > macd_sig.iloc[-1] and
        v > vol_avg.iloc[-1] and
        not kurum_satiyor
    )

    al = (
        c > ema20.iloc[-1] and
        r.iloc[-1] > 50 and
        macd.iloc[-1] > macd_sig.iloc[-1]
    )

    son_yukselis = (c - low.rolling(20).min().iloc[-1]) / max(low.rolling(20).min().iloc[-1], 0.001) * 100

    gec_kaldin = son_yukselis > 25 and r.iloc[-1] > 70

    kar_al = (
        son_yukselis > 15 and
        r.iloc[-1] > 62 and
        c < close.iloc[-2] and
        macd.iloc[-1] < macd_sig.iloc[-1]
    )

    sat = (
        c < ema50.iloc[-1] and
        r.iloc[-1] < 45 and
        macd.iloc[-1] < macd_sig.iloc[-1]
    )

    spek = 0

    # Hacim patlaması
    if v > vol_avg.iloc[-1] * 4:
        spek += 30
    elif v > vol_avg.iloc[-1] * 3:
        spek += 20
    elif v > vol_avg.iloc[-1] * 2:
        spek += 10

    # Para girişi
    if mf.iloc[-1] > 70:
        spek += 20
    elif mf.iloc[-1] > 60:
        spek += 10

    # RSI güçleniyor
    if r.iloc[-1] > 60 and r.iloc[-1] > r.iloc[-2]:
        spek += 10

    # Son 5 günlük kırılım
    if c > high.tail(5).max():
        spek += 20

    # Günlük yükseliş
    gunluk_getiri = ((c / close.iloc[-2]) - 1) * 100

    if gunluk_getiri > 8:
        spek += 20
    elif gunluk_getiri > 5:
        spek += 10

    # Son 20 günün zirvesine yakın mı?
    if c >= high.rolling(20).max().iloc[-1] * 0.97:
        spek += 10

    spek = min(100, spek)
    puan = 0
    puan += 25 if dipten_al else 0
    puan += 20 if kurum_topluyor else 0
    puan += 15 if guvenli_al else 0
    puan += 5 if v > vol_avg.iloc[-1] * 2 else 0
    puan += 5 if 50 < r.iloc[-1] < 70 else 0
    puan += 5 if c > ema20.iloc[-1] else 0
    puan += 5 if ema20.iloc[-1] > ema50.iloc[-1] else 0
    puan += 5 if c > ema200.iloc[-1] else 0
    puan += 5 if macd.iloc[-1] > macd_sig.iloc[-1] else 0

    if kurum_satiyor:
        sinyal = "KURUM SATIYOR"
    elif sat:
        sinyal = "SAT"
    elif kar_al:
        sinyal = "KÂR AL"
    elif gec_kaldin:
        sinyal = "GEÇ KALDIN"
    elif dipten_al:
        sinyal = "DİPTEN AL"
    elif kurum_topluyor:
        sinyal = "KURUM TOPLUYOR"
    elif guvenli_al:
        sinyal = "GÜVENLİ AL"
    elif al:
        sinyal = "AL"
    else:
        sinyal = "BEKLE"

    return sinyal, int(puan), spek

def trade_plan(df):
    close = df["Close"]
    high = df["High"]
    low = df["Low"]

    son_fiyat = float(close.iloc[-1])

    destek = float(low.rolling(20).min().iloc[-1])
    direnc1 = float(high.rolling(20).max().iloc[-1])
    direnc2 = float(high.rolling(60).max().iloc[-1])
    ana_hedef = float(high.rolling(120).max().iloc[-1])

    alim_alt = round(destek * 1.02, 2)
    alim_ust = round(son_fiyat, 2)
    stop = round(destek * 0.98, 2)

    hedef1 = round(direnc1, 2)
    hedef2 = round(direnc2, 2)
    ana_hedef = round(ana_hedef, 2)

    risk = max(alim_ust - stop, 0.01)
    odul = max(hedef1 - alim_ust, 0.01)
    risk_odul = round(odul / risk, 2)

    if risk_odul >= 3:
        plan_kalitesi = "İYİ"
    elif risk_odul >= 2:
        plan_kalitesi = "ORTA"
    else:
        plan_kalitesi = "ZAYIF"

    if son_fiyat <= alim_ust:
        alim_zamani = "DESTEK ÜSTÜ ALIM"
    else:
        alim_zamani = "BEKLE"

    if risk_odul >= 3:
        tahmini_sure = "2-6 HAFTA"
    elif risk_odul >= 2:
        tahmini_sure = "1-3 HAFTA"
    else:
        tahmini_sure = "3-10 GÜN"

    return {
        "Son Fiyat": round(son_fiyat, 2),
        "Alım Alt": alim_alt,
        "Alım Üst": alim_ust,
        "Stop": stop,
        "Hedef 1": hedef1,
        "Hedef 2": hedef2,
        "Ana Hedef": ana_hedef,
        "Risk/Ödül": risk_odul,
        "Alım Zamanı": alim_zamani,
        "Tahmini Süre": tahmini_sure,
        "Plan Kalitesi": plan_kalitesi
    }
# =========================
# KARAR
# =========================
def karar(p):
    if p >= 90:
        return "ELMAS"
    if p >= 80:
        return "ÇOK GÜÇLÜ"
    if p >= 70:
        return "AL ADAYI"
    if p >= 60:
        return "TAKİP"
    return "ZAYIF"

# =========================
# TARAMA
# =========================
rows = []

for i, sym in enumerate(symbols, 1):
    print(f"{i}/{len(symbols)} taranıyor: {sym}")

    try:
        df = download_data(sym)

        if df is None:
            print(sym, "veri yok veya yetersiz")
            continue

        daily = df.copy()
        weekly = resample_ohlcv(df, "W")
        monthly = resample_ohlcv(df, "M")

        d_sig, d_puan, d_spek = signal_engine(daily)
        w_sig, w_puan, w_spek = signal_engine(weekly)
        m_sig, m_puan, m_spek = signal_engine(monthly)

        if m_sig == "YETERSİZ":
            toplam_puan = int(d_puan * 0.60 + w_puan * 0.40)
        elif w_sig == "YETERSİZ":
            toplam_puan = int(d_puan)
        else:
            toplam_puan = int(d_puan * 0.50 + w_puan * 0.30 + m_puan * 0.20)
        plan = trade_plan(daily)
        gunluk_getiri = round(((daily["Close"].iloc[-1] / daily["Close"].iloc[-2]) - 1) * 100, 2)

        rows.append({
            "Hisse": sym,
            "Günlük": d_sig,
            "Haftalık": w_sig,
            "Aylık": m_sig,
            "Günlük Puan": d_puan,
            "Haftalık Puan": w_puan,
            "Aylık Puan": m_puan,
            "Günlük Getiri": gunluk_getiri,
            "Toplam Puan": toplam_puan,
            "Spekülasyon": d_spek,
            "Karar": karar(toplam_puan),

            "Son Fiyat": plan["Son Fiyat"],
            "Alım Alt": plan["Alım Alt"],
            "Alım Üst": plan["Alım Üst"],
            "Stop": plan["Stop"],
            "Hedef 1": plan["Hedef 1"],
            "Hedef 2": plan["Hedef 2"],
            "Ana Hedef": plan["Ana Hedef"],
            "Risk/Ödül": plan["Risk/Ödül"],
            "Alım Zamanı": plan["Alım Zamanı"],
            "Tahmini Süre": plan["Tahmini Süre"],
            "Plan Kalitesi": plan["Plan Kalitesi"]
})

    except Exception as e:
        print(sym, "hata:", e)

result = pd.DataFrame(rows)

if result.empty:
    print("Sonuç oluşmadı. Veri kaynağı çalışmamış olabilir.")
else:
    result = result.sort_values("Toplam Puan", ascending=False)

    file_name = f"bist_tum_tarama_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

    adaylar = result[
        result["Günlük"].isin(["DİPTEN AL", "GÜVENLİ AL", "KURUM TOPLUYOR", "AL"])
    ]

    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        result.to_excel(writer, index=False, sheet_name="Tüm Sonuçlar")
        result.head(30).to_excel(writer, index=False, sheet_name="En Güçlü 30")
        adaylar.to_excel(writer, index=False, sheet_name="Bugün Adaylar")

    print("Tarama bitti.")
    print("Dosya hazır:", file_name)

    # files.download(file_name)

    import requests

    TELEGRAM_TOKEN = "8819448065:AAHs_FNHi4zwlLhiUHExKAs7EOco-CaEl0g"
    CHAT_ID = "5920392803"

    def telegram_gonder(mesaj):
        url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"

        data = {
            "chat_id": CHAT_ID,
            "text": mesaj,
            "parse_mode": "HTML"
        }

        r = requests.post(url, data=data)

        print("STATUS =", r.status_code)
        print("CEVAP =", r.text)
if "Risk/Ödül" not in result.columns:
    result["Risk/Ödül"] = 0

if "Spekülasyon" not in result.columns:
    result["Spekülasyon"] = 0

if "Günlük Getiri" not in result.columns:
    result["Günlük Getiri"] = 0

top10 = result[
    (result["Toplam Puan"] >= 40) &
    (result["Spekülasyon"].between(20, 100)) &
    (result["Risk/Ödül"] >= 0) &
    (result["Günlük Getiri"] > 0) &
    (result["Günlük Getiri"] <= 10) &
    (result["Günlük"].isin(["DİPTEN AL", "GÜVENLİ AL", "KURUM TOPLUYOR", "AL"])) &
    (~result["Haftalık"].isin(["SAT", "KÂR AL", "GEÇ KALDIN"]))
].sort_values(
    ["Toplam Puan", "Risk/Ödül", "Spekülasyon"],
    ascending=False
).head(10)

print(type(top10))
print(len(top10))
print(top10.columns.tolist())

mesaj = "🚀 <b>AYDIN BIST ROBOTU - BUGÜNÜN EN GÜÇLÜ 10 HİSSESİ</b>\n\n"
for i, row in top10.iterrows():
    mesaj += f"📌 <b>{row['Hisse']}</b>\n"
    mesaj += f"Günlük: {row['Günlük']}\n"
    mesaj += f"Haftalık: {row['Haftalık']}\n"
    mesaj += f"Aylık: {row['Aylık']}\n"
    mesaj += f"Puan: {row['Toplam Puan']}\n"
    mesaj += f"Karar: {row['Karar']}\n"
    mesaj += f"🔥 Spekülasyon Skoru: {row['Spekülasyon']}/100\n"
    mesaj += f"Risk/Ödül: {row['Risk/Ödül']}\n"

    if row["Spekülasyon"] >= 80:
        mesaj += "🚨 TERA TARZI HAREKET ADAYI\n"
    elif row["Spekülasyon"] >= 60:
        mesaj += "⚠️ DİKKAT ÇEKEN HACİM HAREKETİ\n"

    mesaj += f"\nFiyat: {row['Son Fiyat']}\n"
    mesaj += f"Alım Bölgesi: {row['Alım Alt']} - {row['Alım Üst']}\n"
    mesaj += f"Stop: {row['Stop']}\n"
    mesaj += f"Hedef 1: {row['Hedef 1']}\n"
    mesaj += f"Hedef 2: {row['Hedef 2']}\n"
    mesaj += f"Ana Hedef: {row['Ana Hedef']}\n"
    mesaj += f"Alım Zamanı: {row['Alım Zamanı']}\n"
    mesaj += f"Tahmini Süre: {row['Tahmini Süre']}\n"
    mesaj += f"Plan Kalitesi: {row['Plan Kalitesi']}\n\n"

print("MESAJ UZUNLUK =", len(mesaj))
telegram_gonder(mesaj)
print("Telegram mesajı gönderildi.")


display(result.head(30))

Taranacak hisse sayısı: 431
1/431 taranıyor: A1CAP
2/431 taranıyor: ACSEL


/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_3820/4094712482.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_3820/4094712482.py:89: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_3820/4094712482.py:90: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_3820/4094712482.py:91: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()
/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in 

3/431 taranıyor: ADEL


/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_3820/4094712482.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_3820/4094712482.py:89: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_3820/4094712482.py:90: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_3820/4094712482.py:91: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()
/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in 

4/431 taranıyor: ADESE
5/431 taranıyor: AEFES


...

/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_3820/4094712482.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_3820/4094712482.py:89: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_3820/4094712482.py:90: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_3820/4094712482.py:91: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()
/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in 

430/431 taranıyor: ZOREN
431/431 taranıyor: ZRGYO


/tmp/ipykernel_3820/4094712482.py:87: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_3820/4094712482.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_3820/4094712482.py:89: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_3820/4094712482.py:90: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_3820/4094712482.py:91: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()


Tarama bitti.
Dosya hazır: bist_tum_tarama_20260608_2150.xlsx
<class 'pandas.core.frame.DataFrame'>
10
['Hisse', 'Günlük', 'Haftalık', 'Aylık', 'Günlük Puan', 'Haftalık Puan', 'Aylık Puan', 'Günlük Getiri', 'Toplam Puan', 'Spekülasyon', 'Karar', 'Son Fiyat', 'Alım Alt', 'Alım Üst', 'Stop', 'Hedef 1', 'Hedef 2', 'Ana Hedef', 'Risk/Ödül', 'Alım Zamanı', 'Tahmini Süre', 'Plan Kalitesi']
MESAJ UZUNLUK = 3194
STATUS = 200
CEVAP = {"ok":true,"result":{"message_id":58,"from":{"id":8819448065,"is_bot":true,"first_name":"Ayd\u0131n BIST Robotu","username":"aydin_bist_robotu_bot"},"chat":{"id":5920392803,"first_name":"H\u00fcseyin","last_name":"Ayd\u0131n","type":"private"},"date":1780955438,"text":"\ud83d\ude80 AYDIN BIST ROBOTU - BUG\u00dcN\u00dcN EN G\u00dc\u00c7L\u00dc 10 H\u0130SSES\u0130\n\n\ud83d\udccc AKENR\nG\u00fcnl\u00fck: KURUM TOPLUYOR\nHaftal\u0131k: KURUM TOPLUYOR\nAyl\u0131k: YETERS\u0130Z\nPuan: 59\nKarar: ZAYIF\n\ud83d\udd25 Spek\u00fclasyon Skoru: 40/100\nRisk/\u00d6d\u00fcl: 

,Hisse,Günlük,Haftalık,Aylık,Günlük Puan,Haftalık Puan,Aylık Puan,Günlük Getiri,Toplam Puan,Spekülasyon,...,Alım Alt,Alım Üst,Stop,Hedef 1,Hedef 2,Ana Hedef,Risk/Ödül,Alım Zamanı,Tahmini Süre,Plan Kalitesi
12,AKENR,KURUM TOPLUYOR,KURUM TOPLUYOR,YETERSİZ,65,50,0,3.90,59,40,...,11.13,15.17,10.69,15.47,15.47,15.47,0.07,BEKLE,3-10 GÜN,ZAYIF
309,OZGYO,KURUM TOPLUYOR,KURUM TOPLUYOR,YETERSİZ,65,50,0,8.77,59,70,...,2.11,2.48,2.03,2.50,2.50,2.52,0.04,BEKLE,3-10 GÜN,ZAYIF
161,FLAP,KURUM TOPLUYOR,KURUM TOPLUYOR,YETERSİZ,65,45,0,4.17,57,30,...,12.26,15.00,11.78,16.60,16.60,16.60,0.50,DESTEK ÜSTÜ ALIM,3-10 GÜN,ZAYIF
184,GSDDE,DİPTEN AL,GÜVENLİ AL,YETERSİZ,60,45,0,4.55,54,10,...,11.69,12.41,11.23,17.50,18.88,18.88,4.31,DESTEK ÜSTÜ ALIM,2-6 HAFTA,İYİ
95,CELHA,KURUM TOPLUYOR,GEÇ KALDIN,YETERSİZ,65,35,0,8.94,53,50,...,12.87,17.30,12.37,20.10,20.10,20.10,0.57,DESTEK ÜSTÜ ALIM,3-10 GÜN,ZAYIF
82,BRLSM,GÜVENLİ AL,KURUM TOPLUYOR,YETERSİZ,40,65,0,9.92,50,20,...,16.00,21.38,15.38,25.48,25.48,25.48,0.68,DESTEK ÜSTÜ ALIM,3-10 GÜN,ZAYIF
202,IDGYO,GEÇ KALDIN,KURUM TOPLUYOR,YETERSİZ,40,65,0,7.57,50,50,...,3.37,5.26,3.23,5.30,5.30,5.30,0.02,BEKLE,3-10 GÜN,ZAYIF
363,SOKE,SAT,GÜVENLİ AL,YETERSİZ,55,40,0,2.87,49,0,...,15.80,16.47,15.18,20.86,21.20,21.20,3.40,DESTEK ÜSTÜ ALIM,2-6 HAFTA,İYİ
337,RNPOL,KURUM TOPLUYOR,BEKLE,YETERSİZ,65,20,0,8.73,47,40,...,2.16,2.74,2.08,2.82,3.38,3.38,0.12,BEKLE,3-10 GÜN,ZAYIF
106,CUSAN,GEÇ KALDIN,KURUM TOPLUYOR,YETERSİZ,45,50,0,4.35,47,70,...,22.99,28.80,22.09,29.68,29.68,32.90,0.13,DESTEK ÜSTÜ ALIM,3-10 GÜN,ZAYIF
